In [2]:
import cv2
import mediapipe as mp
import numpy as np
import joblib
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

rf_model = joblib.load('models/gesture_model.joblib')
label_encoder = joblib.load('models/label_encoder.joblib')
MODEL_PATH = "models/gesture_recognizer.task"

BaseOptions = mp.tasks.BaseOptions
GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

In [3]:
options = GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

cap = cv2.VideoCapture(0)

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

True

In [5]:
print("Starting object detection... Press 'q' to quit.")

with GestureRecognizer.create_from_options(options) as recognizer:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("Falha ao capturar frame.")
            break
            
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        timestamp_ms = int(cv2.getTickCount() / cv2.getTickFrequency() * 1000)
        
        result = recognizer.recognize_for_video(mp_image, timestamp_ms)
        
        left_coords = [0.0] * 63
        right_coords = [0.0] * 63
        
        if result.hand_landmarks:
            for idx, hand_landmarks in enumerate(result.hand_landmarks):
                handedness = result.handedness[idx][0].category_name
                
                base_x = hand_landmarks[0].x
                base_y = hand_landmarks[0].y
                base_z = hand_landmarks[0].z
                
                norm_coords = []
                for lm in hand_landmarks:
                    norm_coords.extend([lm.x - base_x, lm.y - base_y, lm.z - base_z])
                    
                if handedness == "Left":
                    left_coords = norm_coords
                elif handedness == "Right":
                    right_coords = norm_coords
                    
                mp_drawing.draw_landmarks(
                    frame, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style())

            features = np.array(left_coords + right_coords).reshape(1, -1)
            
            prediction_idx = rf_model.predict(features)[0]
            predicted_label = label_encoder.inverse_transform([prediction_idx])[0]
            
            probabilities = rf_model.predict_proba(features)[0]
            confidence = probabilities[prediction_idx]
            
            if confidence > 0.60:
                cv2.putText(frame, f"Gesto: {predicted_label} ({confidence:.2f})", (30, 60),
                            cv2.FONT_HERSHEY_DUPLEX, 1.2, (0, 255, 0), 2)
            else:
                cv2.putText(frame, "Gesto: Desconhecido", (30, 60),
                            cv2.FONT_HERSHEY_DUPLEX, 1.2, (0, 0, 255), 2)
        else:
            cv2.putText(frame, "Nenhuma mao detectada", (30, 60),
                        cv2.FONT_HERSHEY_DUPLEX, 1.0, (150, 150, 150), 2)

        cv2.imshow('Custom Gesture Recognition', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
cap.release()
cv2.destroyAllWindows()

Starting object detection... Press 'q' to quit.
